### Copyright Matlantis Corp. as contributors to Matlantis contrib project

##  ポリマーの自由体積計算
- シミュレーションは[こちらの論文](https://www.mdpi.com/2079-3197/7/2/27)を参考に実装した。
- 本notebookではnPTを実行し、構造を緩和させる。

In [ ]:
import json
import sys
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
from ase import units
from ase.io import read
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.md.npt import NPT
from ase.md import MDLogger

from pfcc_extras import show_gui
from pfcc_extras.structure.rotate import convert_atoms_to_upper
from pfcc_extras.structure.connectivity import CollisionDetector
from pfp_api_client import Estimator, ASECalculator

# Fix the random seed for reproducibility. Change the seed value to run with different random numbers.
seed = 42
np.random.seed(seed)

## はじめに
Nose-Hoover熱浴、Parrinello-Rahman圧力浴を用いてnPTアンサンブルのMDを実行するためのサンプルプログラムです。

nPTアンサンブルのMDでは、粒子数(n)、圧力(P)、温度(T)が一定になる原子集団のシミュレーションが実行できます。  
熱浴は、温度(運動エネルギー)を一定に保つために系にエネルギーの与奪を行います。  
このスクリプトではNose-Hooverの熱浴、Parrinello-Rahmanの圧力浴でエネルギーの制御を行います。

## Settings

## Time zone settings
本スクリプトでは出力ファイルに現在時刻を表示します。  
Matlantis環境の時刻はUTCで記録されているため、時差を入力し現在時刻の補正を行います。

日本(JST = UTC+9)の場合は、timedelta(hours=9)と入力してください。

In [ ]:
# --- time difference ---
time_difference = timedelta(hours=9)

now = datetime.now() + time_difference
day = now.strftime("%Y%m%d")
clock = now.strftime("%H%M%S")

## Input/Output settings
入力ファイルと出力ディレクトリ(フォルダ)を指定します。

各計算ごとに一つ出力ディレクトリを用意することを強く推奨します。  
これは同じ階層にファイル出力した場合、取り違えや意図しない上書きが発生するためです。

このスクリプトは、デフォルトでは、
out_parentで指定されたフォルダの中に  
**_{input\_file名(拡張子なし)}_**\_**_{memoで指定した計算条件などの文字}_**\_**_{日付:yyyymmdd}_**\_**_{時刻:hhmmss}_**  
というディレクトリを作成しその中に計算結果を格納します。

なお`memo=None`とした場合は、memo部分が非表示となり、  
**_{input\_file名(拡張子なし)}_**\_**_{日付:yyyymmdd}_**\_**_{時刻:hhmmss}_**  
となります。

出力フォルダ名を独自で定めたい場合、out_dirを独自で設定してください。

In [ ]:
# --- input/output file settings ---'
input_file = Path('output_modeling/UItem_liq.xyz')
out_parent = Path('output_nPT')
memo = None

# Output calculation results to a folder named with the input filename and current time
if memo is None:
    out_dir = out_parent / f'{input_file.stem}_{day}_{clock}'
else:
    out_dir = out_parent / f'{input_file.stem}_{memo}_{day}_{clock}'

log_file = out_dir / f'nPT_NoseHoover_ParrinelloRahman.log'
traj_file = out_dir / f'nPT_NoseHoover_ParrinelloRahman.traj'
setting_file = out_dir / f'calc_settings.json'

## PFP settings

In [ ]:
model_version = 'v8.0.0'
calc_mode = 'R2SCAN_PLUS_D3'

## MD settings
1 ns = 1_000 ps = 1_000_000 fs  
Underscore(_) can be used to represent large number: 1000000 == 1_000_000

In [ ]:
# --- time related settings ---
time_step = 1   # fsec
num_md_steps = 1_000_000

# --- temperature and pressure related settings ---
temperature = 300  # Kelvin
pressure = 1.0  # bar

# --- thermostat and barostat related settings ---
ttime = 20.0   # tau_T thermostat time constant in fsec
pfactor = 2e6  # pressure control parameters. Notice: this is not pressure.

## Logging settings
`file_interval`: 何ステップに一度ログファイルを出力するか  
`time_check_interval`: 何ステップに一度経過時間を確認するか。経過時間確認のコストは無視できるほど小さいため基本は1を指定してください。  
`print_interval_seconds`: 何秒経過した場合に経過時間をNotebookに出力するか。60秒に1回程度の頻度で出力すると、計算所要時間の見積もり等に利用できて便利です。

In [ ]:
file_interval = 100
time_check_interval = 1
print_interval_seconds = 60

## MISC
MDの計算結果に影響を与えない設定集です。

### Cellの取り直し
ASEの仕様によりNPTクラスを用いる際にはスーパーセルが上三角行列でなければ計算ができません。  
このため、特に強い理由がない場合は、`convert_to_upper_triangle=True`を採用してください。  
用意したインプットと座標軸の取り方が変換されます。

### 接触判定
MDの計算前にインプット内で原子の衝突が起きているかどうかを判定します。  
collision_multは標準的な共有結合長の何倍よりも短い結合長を持つ原子ペアが存在すれば接触と判断するかの閾値です。  
connection_multは接触部分を分子として抽出する際にどのくらいの結合長であれば分子と判定するかの閾値です。  

In [ ]:
# Cell reconstruction
convert_to_upper_triange = True

# Collision detection
collision_mult = 0.7
connection_mult = 1.0

## check

In [ ]:
if not input_file.exists():
    raise FileNotFoundError(f'input file: "{input_file}" is not found.')
    
print(f'Input: "{input_file}"')
print(f'Output: "{out_dir}"')

atoms = read(str(input_file))
if convert_to_upper_triange:
    atoms = convert_atoms_to_upper(atoms)
show_gui(atoms)

In [ ]:
detector = CollisionDetector(atoms, collision_mult=collision_mult, connection_mult=connection_mult)
if any(detector.is_colliding()):
    raise RuntimeError(f'Colliding atoms detected in the input. Indices:{detector.get_colliding_indices()}')

In [ ]:
# To extract and show colliding parts as molecules, uncomment the following:
# show_gui(atoms[detector.get_colliding_molecule_indices()])

# To extract and show only colliding atoms, uncomment the following:
# show_gui(atoms[detector.get_colliding_indices()])

## End of settings
# Calculation Part
## Pretreatment

In [ ]:
# === PFP ===
estimator = Estimator(model_version=model_version, calc_mode=calc_mode)
calculator = ASECalculator(estimator)
atoms.calc = calculator

# === make output directory ===
out_dir.mkdir(exist_ok=True, parents=True)

# === write settings ===
settings = {
    "force_field_params":
        {
            "method": "pfp",
            "model_version": model_version,
            "calc_mode": calc_mode,
        },
    "MD_params":
    {
        "time_step": time_step,
        "num_md_steps": num_md_steps,
        "dump_interval": file_interval,
        "ensemble": "nVT",
        "ensemble_params":
            {
                "thermostat": "Nose-Hoover",
                "temperature": temperature,
                "ttime": ttime,
                "barostat": "Parrinello-Rahman",
                "pressure": pressure,
                "pfactor": pfactor,
            }
    },
}

with open(setting_file, 'w') as f:
    json.dump(settings, f)

## Configuration of display functions

In [ ]:
from ase.utils import IOContext

class TimeLogger(IOContext):
    def __init__(self, dyn, atoms, time_difference, max_steps, logfile=sys.stdout, log_interval_seconds=60):
        self.dyn = dyn
        self.atoms = atoms
        self.logfile = logfile
        
        now = datetime.now() + time_difference
        self.start_time = now
        self.log_interval_seconds = log_interval_seconds
        self.time_difference = time_difference
        self.max_steps = max_steps
        self.header = f'{"Steps":>10}, {"Current time":>25}, {"Elapsed time":>20}, {"Estimated remain":>20}\n'
        self.header += '=' * 81 + '\n'
        self.logfile.write(self.header)
        
        steps, days, hours, minutes, seconds = 0, 0, 0, 0, 0
        timestamp = now.strftime('%Y-%m-%d %H:%M:%S')
        elapsed_time_str = f"{int(days)} days {int(hours):02}:{int(minutes):02}:{int(seconds):02}"
        log = f'{steps:>10}, {timestamp:>25}, {elapsed_time_str:>20}, {"Unknown":>20}\n'
        self.logfile.write(log)
        self.cnt = 1
        
    def __del__(self):
        self.close()
        
    def __call__(self):
        now = datetime.now() + time_difference
        if (now - self.start_time).total_seconds() <= self.log_interval_seconds * self.cnt:
            return
        
        steps = self.dyn.nsteps
        timestamp = now.strftime('%Y-%m-%d %H:%M:%S')
        
        elapsed_time = now - self.start_time
        days, remainder = divmod(elapsed_time.total_seconds(), 3600*24)
        hours, remainder = divmod(remainder, 3600)
        minutes, seconds = divmod(remainder, 60)
        elapsed_time_str = f"{int(days)} days {int(hours):02}:{int(minutes):02}:{int(seconds):02}"
        
        estimated_remain = elapsed_time * (self.max_steps - steps) / steps
        r_days, remainder = divmod(estimated_remain.total_seconds(), 3600*24)
        r_hours, remainder = divmod(remainder, 3600)
        r_minutes, r_seconds = divmod(remainder, 60)
        estimated_remain_str = f"{int(r_days)} days {int(r_hours):02}:{int(r_minutes):02}:{int(r_seconds):02}"
        
        log = f'{steps:>10}, {timestamp:>25}, {elapsed_time_str:>20}, {estimated_remain_str:>20}\n'
        
        self.before_time = now
        self.logfile.write(log)
        self.cnt +=1

## Assign initial velocities according to the Maxwell-Boltzmann distribution.

In [ ]:
# rng is the seed to fix random numbers. Added for reproducibility of the results.
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature, force_temp=True, rng=np.random.RandomState(seed))
Stationary(atoms)  # Set zero total momentum to avoid drifting

## Execution of NPT

In [ ]:
dyn = NPT(
    atoms,
    time_step*units.fs,
    temperature_K = temperature,
    ttime = ttime*units.fs,
    pfactor = pfactor * units.GPa * (units.fs**2),
    externalstress=pressure*units.bar,
    loginterval=file_interval,
    trajectory=str(traj_file)
)

# set logger
# dyn.attach(print_dyn, interval=print_interval)
dyn.attach(MDLogger(dyn, atoms, str(log_file), header=True, stress=True, peratom=True, mode="w"), interval=file_interval)
dyn.attach(TimeLogger(dyn, atoms, time_difference, num_md_steps, log_interval_seconds=print_interval_seconds), interval=time_check_interval)

# run MD
dyn.run(num_md_steps)